In [1]:
import pandas as pd
import networkx as nx

from utils.geometry_utils import rebuild_polygons
from utils.graph_utils import build_graph

In [2]:
rooms_df = pd.read_csv("rooms_dataset.csv")
rooms_df = rebuild_polygons(rooms_df)

In [3]:
public_rooms = ["LivingRoom", "DiningRoom", "Entry", "Entry Lobby", "Terrace"]
private_rooms = ["Bedroom", "Bath", "Bathroom", "Master Bedroom"]
service_rooms = ["Kitchen", "Utility", "Laundry", "Pantry"]

In [4]:
def public_private_separation(G):
    if G.number_of_nodes() == 0:
        return 0
    violations = 0
    total_edges = G.number_of_edges()
    if total_edges == 0:
        return 0
    for u, v in G.edges():
        r1 = G.nodes[u]["room_type"]
        r2 = G.nodes[v]["room_type"]
        if (r1 in public_rooms and r2 in private_rooms) or (r2 in public_rooms and r1 in private_rooms):
            violations += 1
    return 1 - (violations / total_edges)  # higher is better separation

In [5]:
def service_area_ratio(G):
    service_count = sum(1 for n, d in G.nodes(data=True) if d["room_type"] in service_rooms)
    total_count = G.number_of_nodes()
    if total_count == 0:
        return 0
    return service_count / total_count

In [6]:
def bathroom_adjacency(G):
    bedrooms = [n for n, d in G.nodes(data=True) if "Bedroom" in d["room_type"]]
    bathrooms = [n for n, d in G.nodes(data=True) if "Bath" in d["room_type"] or "Bathroom" in d["room_type"]]
    if len(bedrooms) == 0 or len(bathrooms) == 0:
        return 0
    adj_count = 0
    for bed in bedrooms:
        for bath in bathrooms:
            if G.has_edge(bed, bath):
                adj_count += 1
    return adj_count / len(bedrooms)

In [7]:
results = []

for plan_id in rooms_df["plan_id"].unique():
    plan_rooms = rooms_df[rooms_df["plan_id"] == plan_id]
    G = build_graph(plan_rooms)
    
    metrics = {}
    metrics["plan_id"] = plan_id
    metrics["public_private_separation"] = public_private_separation(G)
    metrics["service_area_ratio"] = service_area_ratio(G)
    metrics["bathroom_adjacency"] = bathroom_adjacency(G)
    
    results.append(metrics)

In [8]:
functional_features_df = pd.DataFrame(results)
functional_features_df.head()

,plan_id,public_private_separation,service_area_ratio,bathroom_adjacency
0,6044,0.812500,0.100000,0.0
1,2564,1.000000,0.000000,0.0
2,6165,1.000000,0.166667,0.0
3,1021,0.939394,0.000000,1.0
4,5507,0.857143,0.062500,0.0


In [9]:
functional_features_df.to_csv("functional_zoning_features.csv", index=False)